# Auto MPG - predicting fuel efficiency

Quick regression project on the classic auto-mpg dataset. Goal is just to predict mpg from the other specs (cylinders, weight, horsepower etc.) and see how much a plain linear model can explain, then check if regularization helps given how correlated a lot of these features are.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Load the data

In [ ]:
df = pd.read_csv("auto-mpg.csv")
df = df.convert_dtypes()
df.head()

In [ ]:
df.info()

horsepower is coming in as a string column, that's usually a sign there are some "?" values hiding in there for missing data. Checking that next.

In [ ]:
df["horsepower"].unique()

## Cleaning

In [ ]:
# confirmed - horsepower has "?" for missing values instead of NaN
df["horsepower"] = df["horsepower"].replace("?", np.nan)
df["horsepower"] = pd.to_numeric(df["horsepower"])

df.isnull().sum()

In [ ]:
# only a handful of rows missing horsepower, fine to just drop them
df = df.dropna().reset_index(drop=True)
df.shape

## A quick look at the relationships

In [ ]:
df_num = df.select_dtypes(include="number").astype(float)

plt.figure(figsize=(8, 6))
sns.heatmap(df_num.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix")
plt.show()

mpg correlates strongly (negatively) with cylinders, displacement, horsepower and weight - basically bigger/heavier engines get worse mileage, no surprises there.

Problem is those four are also all heavily correlated with each other, which is going to cause multicollinearity issues for a plain linear regression. Keeping that in mind for later, might be worth trying Ridge/Lasso once there's a baseline to compare against.

## Features and target

In [ ]:
feature_cols = [
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model year",
    "origin",
]

X = df[feature_cols]
y = df["mpg"]

# dropping car name - it's just a text label, would need encoding to be useful
# origin is technically categorical (1/2/3 = country) but treating it as
# numeric for now to keep things simple

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

## Baseline: plain linear regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("R2  :", r2_score(y_test, y_pred))

In [ ]:
pd.Series(lr.coef_, index=X_train.columns).sort_values()

# weight looks tiny here but that's just because it's measured in the
# thousands - not really comparable to the other coefficients until
# everything is on the same scale

## Ridge / Lasso / ElasticNet with CV

Given the multicollinearity from the heatmap above, worth checking whether regularization actually helps or makes no real difference. Scaling first since these penalties are scale-sensitive, and letting grid search pick the best model + alpha combo in one go.

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge()),   # placeholder, gets swapped by param_grid below
])

param_grid = [
    {"model": [Ridge()], "model__alpha": [10, 1.0, 0.1, 0.01]},
    {"model": [Lasso(max_iter=10000)], "model__alpha": [10, 1.0, 0.1, 0.01]},
    {"model": [ElasticNet(max_iter=10000)], "model__alpha": [10, 1.0, 0.1, 0.01],
     "model__l1_ratio": [0.1, 0.5, 0.9]},
]

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)

grid.best_params_, grid.best_score_

## Test set performance

In [ ]:
y_pred_best = grid.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_best)
rmse = mean_squared_error(y_test, y_pred_best) ** 0.5
r2 = r2_score(y_test, y_pred_best)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2  : {r2:.2f}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_best, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual MPG")
plt.ylabel("Predicted MPG")
plt.title("Predicted vs actual")
plt.show()

## Which features is the winning model actually using

In [ ]:
best_model = grid.best_estimator_.named_steps["model"]

coeffs = pd.Series(best_model.coef_, index=X_train.columns).sort_values()
coeffs

model year and weight are doing most of the work here - newer, lighter cars get better mileage, which tracks. Displacement usually gets pushed close to zero by Lasso since it's so redundant with weight/horsepower/cylinders, matches the multicollinearity concern from the heatmap earlier.

### Next steps if I come back to this
- try polynomial features on weight/horsepower, relationship might not be fully linear
- look at residuals vs predicted instead of just the actual-vs-predicted scatter, might reveal where the model struggles (older, low-mpg cars maybe)
- one-hot encode origin instead of treating it as numeric
- a tree-based model (random forest / gradient boosting) as a non-linear comparison point